# M2 원인 분리: history-only (rho=0) 일치 대조군 — Dunnhumby seed 42

직전 구매이력 기반 N/V 조건부 M2의 하락 원인을 분리합니다. 새로 학습하는 모형은 `history_only_rho0` 하나입니다.

- 사용자 표현: 구매한 상품의 학습 임베딩을 그래프 연결강도로 모아 정규화한 `H_u`
- `rho=0`: N/V 조건부 변환은 정확히 꺼지며 `E_u^(0)=H_u`
- 일치 조건: full M2와 같은 seed 42, 100 epoch, 64차원, rank 4 모듈 생성, binary graph, uniform negative sampling, plain BPR
- 재사용: 저장된 동일 protocol의 M1@64와 full M2 결과
- 분할: 1~683일 학습, 684~690일 신규상품 역사적 개발평가

이 실행은 성능 개선을 주장하는 최종 평가가 아니라, `사용자 ID 제거`와 `N/V 변환`의 영향을 나누는 seed 42 메커니즘 대조입니다. 유의성과 일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'e8dbac0a9e3e65a75b66043d1fcfb49fb1caec2e'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_history_only_control import (
    configure_history_only_control,
    preflight_summary,
    run_history_only_control,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_history_only_control(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_history_only_matched_control_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
    full_m2_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_history_conditioned_lowrank_historical_screen_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['trained_models'] == ['history_only_rho0']
assert summary['reused_comparators'] == [
    'm1_64', 'm2_history_conditioned_lowrank_transform'
]
assert summary['control']['rho'] == 0.0
assert summary['control']['free_user_id_embedding'] is False
assert summary['control']['layer0_identity'] == 'E_u^(0)=H_u exactly'
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['new_loss_term'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
assert summary['fixed']['validation_or_epoch_selection'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_history_only_control(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['mechanism_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1 / history-only / full M2 원인 분리표:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('메커니즘 판독:', json.dumps(reading, ensure_ascii=False, indent=2))
print('결과 파일:', paths)